# 08 · Parallel branches (fan-out and fan-in)

**Goal:** by the end you can run several nodes at the same time from one node (fan-out), collect their results in
one node (fan-in), and explain why parallel nodes writing the same key need a reducer.

## The idea in plain English

If a node has **several** outgoing edges, LangGraph runs **all** their targets, in the same superstep. That's called
**fan-out**. When several edges lead *into* one node, it runs after those branches have finished. That's
**fan-in**.

We give our calculator one input, `a = 12` and `b = 4`, and run `add`, `subtract`, `multiply` and `divide` at the same
time. Then a `summary` node collects all four answers.

There's one catch. All four branches write to the same `results` key in the same superstep. With the default rule
(replace), LangGraph can't know which of the four values should win, so it raises an error. A reducer that
appends solves it.

**Analogy:** a teacher writes two numbers on the board and four students each do one operation at the same time.
The teacher collects all four answers into one list. Nobody's answer overwrites anyone else's.

## Picture

```mermaid
flowchart LR
    S([START]) --> start_node
    start_node --> add & subtract & multiply & divide
    add & subtract & multiply & divide --> summary
    summary --> E([END])
```

| Superstep | Nodes that run |
|---|---|
| 1 | `start_node` |
| 2 | `add`, `subtract`, `multiply`, `divide` (in parallel) |
| 3 | `summary` |

## Step 1: The state, with a reducer on `results`

Each branch adds one line to `results`, so it gets the `operator.add` reducer from lesson 06. `summary` is written
by one node only, so it needs no reducer.

In [1]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, START, END


class ParallelState(TypedDict):
    a: float
    b: float
    results: Annotated[list[str], operator.add]
    summary: str

## Step 2: The nodes

`start_node` is where the branches split. It has nothing to change, so it returns an empty dict (an empty update is
allowed). Each operation node returns a list holding **one** line. Every node prints its name so we can see the
order things run in.

In [2]:
def start_node(state: ParallelState) -> dict:
    print("superstep 1: start_node")
    return {}


def add(state: ParallelState) -> dict:
    print("superstep 2: add")
    return {"results": [f"{state['a']:g} + {state['b']:g} = {state['a'] + state['b']:g}"]}


def subtract(state: ParallelState) -> dict:
    print("superstep 2: subtract")
    return {"results": [f"{state['a']:g} - {state['b']:g} = {state['a'] - state['b']:g}"]}

The other two operations, and `summary`, which joins every line in `results` with ` | `. It runs once the branches it waits for have finished.

In [3]:
def multiply(state: ParallelState) -> dict:
    print("superstep 2: multiply")
    return {"results": [f"{state['a']:g} * {state['b']:g} = {state['a'] * state['b']:g}"]}


def divide(state: ParallelState) -> dict:
    print("superstep 2: divide")
    return {"results": [f"{state['a']:g} / {state['b']:g} = {state['a'] / state['b']:g}"]}


def summary(state: ParallelState) -> dict:
    print("superstep 3: summary")
    return {"summary": " | ".join(state["results"])}

## Step 3: Fan out and fan in

Four edges **leave** `start_node`: that's the fan-out. Four edges **enter** `summary`: that's the fan-in. Nothing
else is new; it's all `add_edge`.

In [4]:
OPERATIONS = [("add", add), ("subtract", subtract), ("multiply", multiply), ("divide", divide)]

builder = StateGraph(ParallelState)
builder.add_node("start_node", start_node)
builder.add_node("summary", summary)
for name, fn in OPERATIONS:
    builder.add_node(name, fn)
    builder.add_edge("start_node", name)  # fan-out
    builder.add_edge(name, "summary")  # fan-in
builder.add_edge(START, "start_node")
builder.add_edge("summary", END)

graph = builder.compile()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	start_node(start_node)
	summary(summary)
	add(add)
	subtract(subtract)
	multiply(multiply)
	divide(divide)
	__end__([<p>__end__</p>]):::last
	__start__ --> start_node;
	add --> summary;
	divide --> summary;
	multiply --> summary;
	start_node --> add;
	start_node --> divide;
	start_node --> multiply;
	start_node --> subtract;
	subtract --> summary;
	summary --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 4: Run it

Watch the printed lines: one node in superstep 1, four in superstep 2, then `summary` once in superstep 3, after all
four branches have finished.

In [5]:
final = graph.invoke({"a": 12, "b": 4})
print()
print("results:", final["results"])
print("summary:", final["summary"])

superstep 1: start_node
superstep 2: add
superstep 2: divide
superstep 2: multiply
superstep 2: subtract
superstep 3: summary

results: ['12 + 4 = 16', '12 / 4 = 3', '12 * 4 = 48', '12 - 4 = 8']
summary: 12 + 4 = 16 | 12 / 4 = 3 | 12 * 4 = 48 | 12 - 4 = 8


**About the order.** The four results are not in the order we added the edges (`add`, `subtract`, `multiply`,
`divide`). LangGraph merges the updates of one superstep in its own fixed order. Here it came out alphabetical by node
name. If the order matters to you, don't rely on it: sort the list, or store results in a dict keyed by operation
name.

In [6]:
print(sorted(final["results"]))

['12 * 4 = 48', '12 + 4 = 16', '12 - 4 = 8', '12 / 4 = 3']


## Step 5: Without a reducer, parallel writes clash

Here is the same graph with `result: str` and **no** reducer. Each branch writes its own line to `result` in the same
superstep. LangGraph refuses to guess which one should win, and raises `InvalidUpdateError`. Notice that all four
branches did run. The error comes when LangGraph tries to merge their updates.

In [7]:
from langgraph.errors import InvalidUpdateError


class ClashState(TypedDict):
    a: float
    b: float
    result: str  # no reducer!


def make_writer(name: str, symbol: str):
    def node(state: ClashState) -> dict:
        print("superstep 2:", name)
        return {"result": f"{state['a']:g} {symbol} {state['b']:g}"}

    return node

`make_writer` is a small function that *builds* a node function. It saves us writing four almost identical nodes
just to show an error.

In [8]:
builder = StateGraph(ClashState)
builder.add_node("start_node", start_node)
for name, symbol in [("add", "+"), ("subtract", "-"), ("multiply", "*"), ("divide", "/")]:
    builder.add_node(name, make_writer(name, symbol))
    builder.add_edge("start_node", name)
    builder.add_edge(name, END)
builder.add_edge(START, "start_node")
clash = builder.compile()

try:
    clash.invoke({"a": 12, "b": 4})
except InvalidUpdateError as err:
    print("\nInvalidUpdateError:", str(err).splitlines()[0])

superstep 1: start_node
superstep 2: add
superstep 2: divide
superstep 2: multiply
superstep 2: subtract

InvalidUpdateError: At key 'result': Can receive only one value per step. Use an Annotated key to handle multiple values.


## Step 6: Branches of different lengths

Suppose the `divide` branch needs an extra step, `round_quotient`, that adds a rounded version of the quotient. Now
one branch takes two supersteps and the others take one.

With four separate edges into `summary`, LangGraph runs `summary` each time a branch arrives. The short branches
finish in superstep 2, so `summary` runs in superstep 3. Then `round_quotient` finishes, so `summary` runs **again**
in superstep 4. That's no error, just a node running twice, which is easy to miss.

In [9]:
def round_quotient(state: ParallelState) -> dict:
    print("superstep 3: round_quotient")
    return {"results": [f"{state['a']:g} / {state['b']:g} is about {round(state['a'] / state['b'], 2)}"]}

`build_uneven` wires the graph with the extra step on the `divide` branch. Its `wait_for_all` argument picks how the branches are joined into `summary`: four separate edges (`False`) or one edge from a list (`True`).

In [10]:
def build_uneven(wait_for_all: bool):
    builder = StateGraph(ParallelState)
    builder.add_node("start_node", start_node)
    builder.add_node("round_quotient", round_quotient)
    builder.add_node("summary", summary)
    for name, fn in OPERATIONS:
        builder.add_node(name, fn)
        builder.add_edge("start_node", name)
    builder.add_edge("divide", "round_quotient")
    last_nodes = ["add", "subtract", "multiply", "round_quotient"]
    if wait_for_all:
        builder.add_edge(last_nodes, "summary")  # ONE edge that waits for all of them
    else:
        for name in last_nodes:
            builder.add_edge(name, "summary")  # four separate edges
    builder.add_edge(START, "start_node")
    builder.add_edge("summary", END)
    return builder.compile()

First with four separate edges. Count how many times `summary` prints.

In [11]:
final = build_uneven(wait_for_all=False).invoke({"a": 10, "b": 3})
print("\nsummary:", final["summary"])

superstep 1: start_node
superstep 2: add
superstep 2: divide
superstep 2: multiply
superstep 2: subtract
superstep 3: round_quotient
superstep 3: summary
superstep 3: summary

summary: 10 + 3 = 13 | 10 / 3 = 3.33333 | 10 * 3 = 30 | 10 - 3 = 7 | 10 / 3 is about 3.33


The fix is to give `add_edge` a **list** of start nodes: `add_edge(["add", "subtract", "multiply", "round_quotient"],
"summary")`. That's a single edge meaning "run `summary` once, after **all** of these have finished". (The superstep
numbers our nodes print are fixed text, so `summary` still prints `superstep 3` even though it now runs in superstep 4.
What matters is that it prints once.)

In [12]:
final = build_uneven(wait_for_all=True).invoke({"a": 10, "b": 3})
print("\nsummary:", final["summary"])

superstep 1: start_node
superstep 2: add
superstep 2: divide
superstep 2: multiply
superstep 2: subtract
superstep 3: round_quotient
superstep 3: summary

summary: 10 + 3 = 13 | 10 / 3 = 3.33333 | 10 * 3 = 30 | 10 - 3 = 7 | 10 / 3 is about 3.33


## What just happened

- **Step 3** added four edges out of `start_node` and four edges into `summary`. The diagram shows the diamond shape.
- **Step 4** printed `start_node`, then the four operation nodes, then `summary` once. `results` held all four
  lines, merged by the reducer, in alphabetical order (`add`, `divide`, `multiply`, `subtract`).
- **Step 5** removed the reducer. All four branches ran, then the merge failed with `InvalidUpdateError: At key
  'result': Can receive only one value per step.`
- **Step 6** made one branch longer. With separate edges, `summary` printed twice and its first run missed the rounded
  quotient. With one list edge, `summary` ran once, after every branch, and included all five lines.

## Common mistakes

1. **Parallel nodes write the same key with no reducer.**
   `InvalidUpdateError: At key 'result': Can receive only one value per step. Use an Annotated key to handle
   multiple values.`
   Fix: add a reducer, as in `Annotated[list[str], operator.add]`.
2. **Branches of different lengths joined with separate edges.** There's **no error**, but the join node runs more
   than once, and its first run sees only part of the results (Step 6).
   Fix: `builder.add_edge([...all last nodes...], "join_node")` so it waits for all of them.
3. **Relying on the order of parallel results.** The order follows LangGraph's own rule, not the order you wrote the
   edges or the order the branches finished.
   Fix: sort the list, or store results in a dict keyed by operation name.

## Try it

1. Add a fifth parallel branch, `power`, that adds `"12 ** 4 = 20736"` to `results`. Rebuild the Step 3 graph and
   check the summary has five parts.
2. Change the state so each branch adds its **number** to `total: Annotated[float, operator.add]` instead of a line
   of text, and `summary` is removed. For `a=12, b=4`, what is `total`? (Hint: `16 + 8 + 48 + 3`.)

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Parallel branches**: nodes that run in the same superstep because they were reached at the same time.
- **Fan-out**: one node with several outgoing edges, so all their targets run in parallel.
- **Fan-in**: several branches leading into one node, which runs after them.
- **`InvalidUpdateError`**: raised when nodes in the same superstep write one key that has no reducer (also when a
  node returns something that isn't a dict, lesson 03).
- **Superstep** (from lesson 07): one round of a run. Parallel nodes share a superstep.